# 04 - CPT → SFT LoRA Training Pipeline

This notebook trains task-specific LoRA adapters after Continual Pre-Training.

For each random seed:

1. Load the base model.
2. Load the CPT LoRA adapter.
3. Merge the CPT adapter into the base model using `merge_and_unload()` only in memory.
4. Train a new LoRA adapter for:
   - Logs classification;
   - QA generation.
5. Save only the resulting SFT LoRA adapter.

In [ ]:
import exp_common as E

RUN_SEEDS = [777]  # then [123], [42]
MAX_SEQ_LENGTH = 2048
SFT_EPOCHS = 1.0
SFT_MAX_STEPS = -1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 2e-4
TASKS = {
    'logs_on_cpt': {
        'train': E.PREPARED_DIR / 'logs_train_sft.jsonl',
        'val': E.PREPARED_DIR / 'logs_val_sft.jsonl',
    },
    'qa_on_cpt': {
        'train': E.PREPARED_DIR / 'qa_train_sft.jsonl',
        'val': E.PREPARED_DIR / 'qa_val_sft.jsonl',
    },
}

MULTITASK_TRAIN = E.PREPARED_DIR / 'multitask_train_sft.jsonl'

In [ ]:
for task, paths in TASKS.items():
    train_rows = E.read_jsonl(paths['train'])
    val_rows = E.read_jsonl(paths['val'])
    for seed in RUN_SEEDS:
        cpt_adapter = E.ADAPTERS_DIR / 'cpt' / f'seed_{seed}' / 'adapter'
        out = E.ADAPTERS_DIR / task / f'seed_{seed}'
        print('Training', task, 'seed', seed, 'on CPT adapter', cpt_adapter)
        E.train_lora_adapter(
            train_rows, val_rows, out, seed=seed, max_seq_length=MAX_SEQ_LENGTH,
            epochs=SFT_EPOCHS, max_steps=SFT_MAX_STEPS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR,
            base_adapter_to_merge=cpt_adapter,
            lora_r=16, lora_alpha=32, lora_dropout=0.05,
        )
print('Done')

# Multitask

In [ ]:
import exp_common as E

RUN_SEEDS = [42, 123, 777] 
MAX_SEQ_LENGTH = 2048
SFT_EPOCHS = 1.0
SFT_MAX_STEPS = -1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 5e-5 

TASKS = {
    'logs': {
        'train': E.PREPARED_DIR / 'logs_train_sft.jsonl',
        'val': E.PREPARED_DIR / 'logs_val_sft.jsonl',
    },
    'qa': {
        'train': E.PREPARED_DIR / 'qa_train_sft.jsonl',
        'val': E.PREPARED_DIR / 'qa_val_sft.jsonl',
    },
}

MULTITASK_TRAIN = E.PREPARED_DIR / 'multitask_train_sft.jsonl'

In [ ]:
multi_train = E.read_jsonl(MULTITASK_TRAIN)
logs_val = E.read_jsonl(E.PREPARED_DIR / 'logs_val_sft.jsonl')
qa_val = E.read_jsonl(E.PREPARED_DIR / 'qa_val_sft.jsonl')
multi_val = logs_val + qa_val

print(f"Multi-task on CPT: train={len(multi_train)}, val={len(multi_val)}")
print(f"  Val contains: Logs={len(logs_val)}, QA={len(qa_val)}")

for seed in RUN_SEEDS:
    cpt_adapter = E.ADAPTERS_DIR / 'cpt' / f'seed_{seed}' / 'adapter'   
    out_dir = E.ADAPTERS_DIR / 'multitask_on_cpt' / f'seed_{seed}'
    print('Training multitask_on_cpt seed ', seed, ' on CPT adapter ', cpt_adapter)
    E.train_lora_adapter(
        multi_train, multi_val, out_dir, seed=seed,
        max_seq_length=MAX_SEQ_LENGTH, epochs=SFT_EPOCHS,
        max_steps=SFT_MAX_STEPS, per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR,
        warmup_ratio=0.3,
        base_adapter_to_merge=cpt_adapter,
        lora_r=16, lora_alpha=32, lora_dropout=0.05,
    )